<!-- colab-badge -->
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex07.1-poisson/Ex07.1_00_environment_check.ipynb)

*Open this notebook in Google Colab. Its first code cell fetches the set's library files from the public course repository, so nothing needs uploading.*

<!-- course-header v3 -->

**Deep Learning for Engineering** · MSc, Aalborg University · 2026

Developed by **Remus Teodorescu** (ret@et.aau.dk), with support from Research Assistant **Noman Khan** (nomank@energy.aau.dk).

*Reference texts — read for the theory. Used as an **inspirational source** for this course, not as a source of its code:*

- Liu, *PINN with Python*, 2025.
- Raissi, Perdikaris & Karniadakis, *Physics-informed neural networks*, J. Comput. Phys. 378 (2019) 686–707.
- Prince, *Understanding Deep Learning*, MIT Press 2023.

Every notebook in this course has been **written and rewritten by the authors named above**. The code, the problems, the data and the exposition are **original to this course** and are not derived from any publisher's code listings or companion notebooks. Where notation matches a textbook's it is the standard notation of the field, and where an idea is a named author's it is cited as theirs in the text.

See `docs/PROVENANCE.md` for what each reference is cited for, set by set.

---

# Ex_07.1 · Notebook 00 — Environment Check

**Paired with L7.1 · Fundamentals of PINNs**

Run this first, top to bottom. Nothing to write: every cell is complete. Its
job is to establish, before you spend an hour on anything else, that

* `torch`, `numpy` and `matplotlib` import and are recent enough;
* the three modules beside this notebook load — `course_core`, `pinn_core`,
  `problem`;
* automatic differentiation gives you the second derivatives a PDE residual
  needs, checked against an analytic answer;
* the samplers put points where you expect;
* the manufactured problem is self-consistent — the source really is the one
  that produces the stated exact solution.

If a cell fails here, fix it before going on.

---

## 0 · Versions

In [ ]:
# files-cell v1 ----------------------------------------------------------
# This set's library files must sit beside the notebook. Locally they
# already do. On Google Colab, where a notebook opens on its own, they are
# fetched from the public course repository. Run this cell first.
import os, urllib.request
FILES = ['course_core.py', 'pinn_core.py', 'problem.py']
URL = "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex07.1-poisson/"
for f in FILES:
    if not os.path.exists(f):
        urllib.request.urlretrieve(URL + f, f)
        print("fetched", f)
print("files ready:", ", ".join(FILES))


In [ ]:
import sys
import numpy as np
import matplotlib
import torch

print("python     ", sys.version.split()[0])
print("numpy      ", np.__version__)
print("matplotlib ", matplotlib.__version__)
print("torch      ", torch.__version__)
print("cuda        available:", torch.cuda.is_available(), " (not needed)")

**What you should see.** Four version numbers. Any Python from 3.9 and any
PyTorch from 2.0 will do, and **no GPU is required anywhere in Part 2** — every
problem in these exercises is small enough that moving data to a GPU costs more
than it saves.

---

## 1 · The three modules

Every Part 2 exercise has the same three files beside it. The first two are
identical in every set; only the third changes.

| | |
|---|---|
| `course_core.py` | shared by the whole course — `set_seed`, `MLP`, `to_tensor`, `check` |
| `pinn_core.py` | the PDE machinery — `grad`, `d2`, samplers, `train_two_stage` |
| `problem.py` | **this** problem — geometry, exact solution, source, plots |

In [ ]:
# --- setup: every Part 2 notebook opens with this cell ------------------
# Needs course_core.py, pinn_core.py and problem.py beside this notebook.
# On Colab the files cell above fetched them from the public course repository.
import os
for f in ("course_core.py", "pinn_core.py", "problem.py"):
    assert os.path.exists(f), f"{f} is missing - run the files cell above first"

from pinn_core import *                                  # noqa: F401,F403
import problem as pb
import numpy as np, torch, matplotlib.pyplot as plt

set_seed(88)
print("device:", DEVICE, " dtype:", torch.get_default_dtype())

**What you should see.** `device: cpu` on most machines and
`dtype: torch.float64`.

Double precision is deliberate and it is not the deep-learning default. A
second derivative of a network is a difference of differences; in float32 the
residual can be dominated by rounding long before it is dominated by the model,
and you would be optimising noise. The cost is roughly a factor of two on CPU.

---

## 2 · The problem

A slot in the stator of an electrical machine, packed with an impregnated
copper winding. The current heats it; the slot walls sit at the temperature of
the surrounding iron.

In [ ]:
pb.describe_problem()

**What you should see.**

```
  slot            : 10 x 20 mm   (half-width 5 mm, half-depth 10 mm)
  bundle k_eff    : 0.70 W/m.K
  wall            : 90 C
  peak rise       : 20 K   -> hot spot 110 C
  source q        : 0.00 .. 2.13 MW/m^3   (mean 0.93)
  non-negative    : True
  implied J       : 10.4 A/mm^2   in the copper   (machines run 5-20)
```

Two of those lines are there to keep the exercise honest.

**`non-negative: True`** — the manufactured source is heating everywhere, as
ohmic dissipation must be. It is easy to invent a tidy exact solution whose
implied source quietly requires heat to be *removed* from part of the domain.
That is a mathematics exercise wearing an engineering costume.

**`implied J: 10.4 A/mm²`** — the current density this problem corresponds to.
Machines run between about 5 and 20, so this one is real. Note what makes it
work: the bundle's effective conductivity is 0.70 W/m·K, not copper's 400. Try
the same slot in solid copper and the current needed for a 20 K rise is
physically impossible — which is exactly why slot hot spots, and not busbar hot
spots, are what limit a machine.

---

## 3 · Automatic differentiation, checked

A PDE residual is made of derivatives of the network with respect to its
*inputs*. Before trusting one, check the machinery against something you can
differentiate by hand.

In [ ]:
# u(x, y) = sin(3x) * exp(-2y)  ->  u_xx = -9u,  u_yy = 4u,  lap = -5u
pts = to_tensor(np.random.default_rng(0).uniform(-1, 1, (200, 2)), requires_grad=True)
u = torch.sin(3 * pts[:, 0:1]) * torch.exp(-2 * pts[:, 1:2])

u_xx = d2(u, pts, 0)
u_yy = d2(u, pts, 1)

check("u_xx = -9u", to_numpy(u_xx), -9.0 * to_numpy(u), tol=1e-9)
check("u_yy = +4u", to_numpy(u_yy),  4.0 * to_numpy(u), tol=1e-9)
check("laplacian = -5u", to_numpy(u_xx + u_yy), -5.0 * to_numpy(u), tol=1e-9)

g = grad(u, pts)
print("\ngrad returns one column per input:", tuple(g.shape))

**What you should see.** Three `PASS` lines with errors around 1e-15, and
`(200, 2)` for the gradient shape.

Two things worth remembering from this cell. `to_tensor(..., requires_grad=True)`
is what makes the coordinates differentiable — omit it and `grad` returns
`None`, which is the single most common first error in Part 2. And `grad`
returns a column per input, so on a space–time problem `grad(u, xyt)[:, 2:3]`
is the time derivative.

---

## 4 · Sampling the slot

In [ ]:
interior = interior_points(600, pb.DOMAIN, method="lhs", seed=1)
boundary = boundary_points(40, pb.DOMAIN, seed=1)

print("interior", interior.shape, " boundary", boundary.shape)
check_shape("interior", interior, (600, 2))
check_shape("boundary", boundary, (160, 2))

on_wall = (np.isclose(np.abs(boundary[:, 0]), pb.A_HALF)
           | np.isclose(np.abs(boundary[:, 1]), pb.B_HALF))
print("every boundary point is on a wall:", bool(on_wall.all()))

fig, ax = plt.subplots(figsize=(4.2, 6.0))
ax.plot(interior[:, 0] * 1e3, interior[:, 1] * 1e3, ".", ms=3,
        color="#1f77b4", label="interior — PDE residual")
ax.plot(boundary[:, 0] * 1e3, boundary[:, 1] * 1e3, ".", ms=5,
        color="#d94f2b", label="boundary — Dirichlet")
ax.set_aspect("equal"); ax.set_xlabel("x  [mm]"); ax.set_ylabel("y  [mm]")
ax.set_title("Where the loss is evaluated")
ax.legend(frameon=False, fontsize=8, loc="upper right")
plt.show()

**What you should see.** Two `PASS` lines, `True`, and a slot outline filled
with blue points and ringed in orange.

The interior points are a **Latin hypercube**, not a uniform draw: each axis is
cut into as many strata as there are points and one point lands in each. Plain
uniform sampling leaves gaps and clumps at the same cost, and a residual
notices gaps.

Note the two sets are sampled separately and mean different things. The
interior points carry the PDE; the boundary points carry the condition on the
walls. Confusing them is how people end up enforcing the equation on the
boundary and the boundary condition nowhere.

---

## 5 · Is the manufactured problem self-consistent?

`problem.py` claims that `q` is exactly the source that produces `θ`. Check it
rather than believe it — by differentiating the stated solution and seeing
whether the equation closes.

In [ ]:
X, Y, pts_np = grid_points(161, 161, pb.DOMAIN)
x, y = pts_np[:, 0], pts_np[:, 1]

h = 1e-4                       # central differences; see the note below
lap = ((pb.theta_exact(x + h, y) - 2 * pb.theta_exact(x, y)
        + pb.theta_exact(x - h, y)) / h ** 2
       + (pb.theta_exact(x, y + h) - 2 * pb.theta_exact(x, y)
          + pb.theta_exact(x, y - h)) / h ** 2)

residual = pb.K_EFF * lap + pb.source(x, y)
q_scale = pb.source(x, y).max()

print(f"max |k_eff * lap(theta) + q| = {np.abs(residual).max():.3e} W/m^3")
print(f"relative to the peak source  = {np.abs(residual).max() / q_scale:.2e}")
print()
print("theta on the four walls, max |.|:")
for name, xs, ys in [
        ("x = -a", np.full(200, -pb.A_HALF), np.linspace(-pb.B_HALF, pb.B_HALF, 200)),
        ("x = +a", np.full(200,  pb.A_HALF), np.linspace(-pb.B_HALF, pb.B_HALF, 200)),
        ("y = -b", np.linspace(-pb.A_HALF, pb.A_HALF, 200), np.full(200, -pb.B_HALF)),
        ("y = +b", np.linspace(-pb.A_HALF, pb.A_HALF, 200), np.full(200,  pb.B_HALF))]:
    print(f"   {name}: {np.abs(pb.theta_exact(xs, ys)).max():.2e} K")

**What you should see.** A relative residual around **1e-12** and four exact
zeros on the walls.

That 1e-12 is the *finite-difference formula*, not the problem. Shrink `h` and
it gets worse, not better: below about 1e-6 the subtraction of two nearly equal
numbers loses more precision than the smaller step gains. Try it — change `h`
to `1e-7` and watch the residual grow by six orders of magnitude.

Which is a preview of why this course uses automatic differentiation. It is not
an approximation with a step size to tune; it applies the chain rule to the
operations the network actually performed, and is exact to machine precision.

---

## 6 · What you are aiming at

In [ ]:
theta = pb.theta_exact(x, y)

fig, axes = plt.subplots(1, 2, figsize=(9.4, 6.2))
pb.plot_field(theta, ax=axes[0], title="exact θ — the target")
pb.plot_source(ax=axes[1])
plt.tight_layout(); plt.show()

i = np.argmax(theta)
print(f"hot spot: {theta.max():.2f} K above the wall, "
      f"at x = {x[i]*1e3:+.2f} mm, y = {y[i]*1e3:+.2f} mm")
print(f"absolute temperature there: {pb.T_WALL + theta.max():.1f} C")

**What you should see.** A temperature field peaking near the middle but pushed
slightly toward +x, and a source shaped like it.

The skew is deliberate. A symmetric field can be fitted well by a network that
has only learned "hot in the middle, cold at the edges"; the offset means the
model has to get the *shape* right, and it removes the symmetry a lazy solution
could exploit. Real slots are hotter toward the closed end for the same reason
the manufactured one is: less of the heat path leads anywhere useful.

---

## 7 · Ready

You have checked the tools, the derivatives, the samplers and the problem
itself. Nothing below this point in Ex_07.1 depends on anything you have not
just verified.

Next: **notebook 01**, where the boundary condition is a penalty in the loss
and you find out what that costs.